# Problem 4: Insurance Claim Fraud Detection with Google TabFM

## Problem
Some insurance claims are intentionally false or exaggerated, increasing insurer losses.

## What we do
Predict whether an incoming claim is suspicious and should be investigated.

## Business goals
- Prioritize investigations
- Reduce fraudulent payouts
- Speed up approval of genuine claims

This notebook is built as a production-style walkthrough: reproducible setup, web download, robust preprocessing, TabFM + baseline comparison, cost-aware thresholds, and persisted artifacts.

## 0) Why this design

### Dataset
We use the widely used `insurance_claims.csv` fraud dataset (target: `fraud_reported`) downloaded directly from web:
`https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv`

### Typical feature coverage in this dataset
- Claim amount (`total_claim_amount`, `injury_claim`, `property_claim`, `vehicle_claim`)
- Accident context (`incident_type`, `collision_type`, `incident_severity`, witnesses)
- Vehicle profile (`auto_make`, `auto_model`, `auto_year`)
- Customer/policy history proxies (`months_as_customer`, policy details, prior financial indicators)

### Modeling approach
- Baseline: XGBoost
- Foundation model: Google TabFM (3 variants)
- Evaluation: PR-AUC, recall, calibration, and investigation-cost-aware thresholding

## 1) Reproducible setup

Optional environment variables:

- `TABFM_DEVICE=auto|cpu|cuda`
- `INS_DATA_URL=https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv`
- `INS_SAMPLE_TRAIN_ROWS=0` (0 means full split)
- `INS_SAMPLE_EVAL_ROWS=0` (0 means full split)
- `INS_MIN_SAMPLE_ROWS=200`
- `TABFM_CONTEXT_MAX_ROWS=700`
- `TABFM_EVAL_MAX_ROWS=0` (0 means full val/test for comparison)
- `TABFM_FAST_MODE=0|1`
- `TABFM_CHECKPOINT_PATH=/abs/path/to/pytorch_model.bin` (optional)

License note:
- TabFM code is Apache-2.0.
- TabFM released weights are non-commercial; review terms before production commercialization.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import torch
from loguru import logger
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TABFM_DEVICE_PREF = os.getenv('TABFM_DEVICE', 'auto').lower().strip()
if TABFM_DEVICE_PREF not in {'auto', 'cpu', 'cuda'}:
    raise ValueError(f'Unsupported TABFM_DEVICE={TABFM_DEVICE_PREF}')

DATA_URL = os.getenv('INS_DATA_URL', 'https://raw.githubusercontent.com/mwitiderrick/insurancedata/master/insurance_claims.csv').strip()
SAMPLE_TRAIN_ROWS = int(os.getenv('INS_SAMPLE_TRAIN_ROWS', '0'))
SAMPLE_EVAL_ROWS = int(os.getenv('INS_SAMPLE_EVAL_ROWS', '0'))
MIN_SAMPLE_ROWS = int(os.getenv('INS_MIN_SAMPLE_ROWS', '200'))
TABFM_CONTEXT_MAX_ROWS = int(os.getenv('TABFM_CONTEXT_MAX_ROWS', '700'))
TABFM_EVAL_MAX_ROWS = int(os.getenv('TABFM_EVAL_MAX_ROWS', '0'))
TABFM_FAST_MODE = os.getenv('TABFM_FAST_MODE', '0').strip() == '1'

if SAMPLE_TRAIN_ROWS != 0 and SAMPLE_TRAIN_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(f'INS_SAMPLE_TRAIN_ROWS must be 0 or > {MIN_SAMPLE_ROWS}')
if SAMPLE_EVAL_ROWS != 0 and SAMPLE_EVAL_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(f'INS_SAMPLE_EVAL_ROWS must be 0 or > {MIN_SAMPLE_ROWS}')
if TABFM_CONTEXT_MAX_ROWS <= 300:
    raise ValueError('TABFM_CONTEXT_MAX_ROWS must be > 300.')
if TABFM_EVAL_MAX_ROWS != 0 and TABFM_EVAL_MAX_ROWS <= 100:
    raise ValueError('TABFM_EVAL_MAX_ROWS must be 0 or > 100.')


def resolve_tabfm_device(preference: str) -> str:
    if preference == 'auto':
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    if preference == 'cuda' and not torch.cuda.is_available():
        logger.warning('TABFM_DEVICE=cuda requested but CUDA unavailable; falling back to cpu')
        return 'cpu'
    return preference


def find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'pyproject.toml').exists():
            return cand
    raise RuntimeError('Could not find project root (pyproject.toml not found).')


PROJECT_ROOT = find_project_root(Path.cwd())
PROBLEM_ROOT = PROJECT_ROOT / 'problems' / 'problem4_insurance_claim_fraud'
DATA_DIR = PROBLEM_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROBLEM_ROOT / 'artifacts'
MODEL_CACHE_DIR = PROBLEM_ROOT / 'data' / 'models' / 'google-tabfm-1.0.0-pytorch'

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = resolve_tabfm_device(TABFM_DEVICE_PREF)

logger.info('Project root: {}', PROJECT_ROOT)
logger.info('Problem root: {}', PROBLEM_ROOT)
logger.info('Raw data dir: {}', DATA_DIR)
logger.info('Artifacts dir: {}', ARTIFACT_DIR)
logger.info('TabFM device preference={} effective={}', TABFM_DEVICE_PREF, DEVICE)
logger.info('Sample train/eval rows: {}/{} (0 means full)', SAMPLE_TRAIN_ROWS, SAMPLE_EVAL_ROWS)
logger.info('TabFM context max rows: {}', TABFM_CONTEXT_MAX_ROWS)
logger.info('TabFM eval max rows: {} (0 means full)', TABFM_EVAL_MAX_ROWS)
logger.info('TabFM fast mode: {}', TABFM_FAST_MODE)

sns.set_theme(style='whitegrid')

## 2) Download insurance claims dataset from web (idempotent)

In [ ]:
DATA_CSV = DATA_DIR / 'insurance_claims.csv'

if DATA_CSV.exists() and DATA_CSV.stat().st_size > 0:
    logger.info('Using existing file {} ({:.1f} KB)', DATA_CSV, DATA_CSV.stat().st_size / 1024)
else:
    logger.info('Downloading from {} ...', DATA_URL)
    urllib.request.urlretrieve(DATA_URL, DATA_CSV)
    logger.info('Saved {} ({:.1f} KB)', DATA_CSV, DATA_CSV.stat().st_size / 1024)

DATA_CSV

## 3) Load and clean raw data

In [ ]:
raw_df = pd.read_csv(DATA_CSV)

# Common cleanup for this dataset.
df = raw_df.copy()
df = df.replace('?', np.nan)

# Remove fully-empty accidental columns if present.
empty_cols = [c for c in df.columns if df[c].isna().all()]
if empty_cols:
    logger.warning('Dropping fully-empty columns: {}', empty_cols)
    df = df.drop(columns=empty_cols)

if 'fraud_reported' not in df.columns:
    raise ValueError('Expected target column fraud_reported not found')

# Add stable claim id for traceability.
df.insert(0, 'claim_id', np.arange(len(df), dtype=np.int64))

# Parse target.
df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})
if df['fraud_reported'].isna().any():
    bad_vals = sorted(df.loc[df['fraud_reported'].isna(), 'fraud_reported'].dropna().unique().tolist())
    raise ValueError(f'Unexpected target values in fraud_reported: {bad_vals}')
df['fraud_reported'] = df['fraud_reported'].astype(int)

logger.info('Shape after cleanup: {}', df.shape)
logger.info('Columns: {}', df.columns.tolist())

class_counts = df['fraud_reported'].value_counts().sort_index()
logger.info('Class distribution: {}', class_counts.to_dict())
logger.info('Fraud rate: {:.2%}', df['fraud_reported'].mean())

pd.DataFrame([
    {'metric': 'rows', 'value': len(df)},
    {'metric': 'columns', 'value': df.shape[1]},
    {'metric': 'fraud_rate', 'value': float(df['fraud_reported'].mean())},
    {'metric': 'fraud_count', 'value': int(class_counts.get(1, 0))},
    {'metric': 'nonfraud_count', 'value': int(class_counts.get(0, 0))},
])

In [ ]:
plt.figure(figsize=(6, 4))
plot_counts = df['fraud_reported'].value_counts().rename(index={0: 'genuine', 1: 'suspicious'})
sns.barplot(x=plot_counts.index, y=plot_counts.values, palette='deep')
plt.title('Target Distribution')
plt.xlabel('Claim type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4) Feature engineering (claims + customer + vehicle)

We build leakage-safe engineered features from fields available at claim intake.

In [ ]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    # Dates.
    out['policy_bind_date'] = pd.to_datetime(out['policy_bind_date'], errors='coerce')
    out['incident_date'] = pd.to_datetime(out['incident_date'], errors='coerce')

    out['policy_tenure_days'] = (out['incident_date'] - out['policy_bind_date']).dt.days
    out['incident_month'] = out['incident_date'].dt.month
    out['incident_weekday'] = out['incident_date'].dt.dayofweek

    # Vehicle age proxy at incident time.
    incident_year = out['incident_date'].dt.year
    out['vehicle_age_at_incident'] = incident_year - pd.to_numeric(out['auto_year'], errors='coerce')

    # Claim amount transforms.
    out['total_claim_log1p'] = np.log1p(pd.to_numeric(out['total_claim_amount'], errors='coerce').clip(lower=0))

    claim_components = ['injury_claim', 'property_claim', 'vehicle_claim']
    for c in claim_components:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    total_claim = pd.to_numeric(out['total_claim_amount'], errors='coerce').replace(0, np.nan)
    out['injury_claim_ratio'] = out['injury_claim'] / total_claim
    out['property_claim_ratio'] = out['property_claim'] / total_claim
    out['vehicle_claim_ratio'] = out['vehicle_claim'] / total_claim

    # Convert selected numeric-looking fields.
    numeric_candidates = [
        'months_as_customer', 'age', 'policy_deductable', 'policy_annual_premium',
        'umbrella_limit', 'capital-gains', 'capital-loss', 'incident_hour_of_the_day',
        'number_of_vehicles_involved', 'bodily_injuries', 'witnesses', 'total_claim_amount',
    ]
    for c in numeric_candidates:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')

    # Hash-like/high leakage identifier columns: keep in metadata, not model features.
    return out


feature_df = engineer_features(df)
feature_df[['claim_id', 'months_as_customer', 'total_claim_amount', 'policy_tenure_days', 'vehicle_age_at_incident', 'fraud_reported']].head()

## 5) Stratified split with optional row caps

In [ ]:
TARGET_COL = 'fraud_reported'
META_COLS = ['claim_id', 'total_claim_amount', 'incident_date', 'policy_bind_date']
DROP_LEAKY_OR_ID = ['claim_id', 'policy_number', 'insured_zip', 'incident_location']


def cap_split_rows(df_split: pd.DataFrame, max_rows: int, seed: int, target_col: str, min_pos: int = 50) -> pd.DataFrame:
    if max_rows == 0 or len(df_split) <= max_rows:
        return df_split.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    pos = df_split[df_split[target_col] == 1]
    neg = df_split[df_split[target_col] == 0]

    if len(pos) == 0:
        return df_split.sample(n=max_rows, random_state=seed).reset_index(drop=True)

    desired_pos = min(len(pos), max(min_pos, int(round(max_rows * len(pos) / len(df_split)))))
    desired_pos = min(desired_pos, max_rows - 1)
    desired_neg = max_rows - desired_pos

    pos_sample = pos.sample(n=desired_pos, random_state=seed)
    neg_sample = neg.sample(n=desired_neg, random_state=seed)
    return pd.concat([pos_sample, neg_sample], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)


train_full, holdout = train_test_split(
    feature_df,
    test_size=0.30,
    random_state=SEED,
    stratify=feature_df[TARGET_COL],
)
val_full, test_full = train_test_split(
    holdout,
    test_size=0.50,
    random_state=SEED,
    stratify=holdout[TARGET_COL],
)

train_df = cap_split_rows(train_full, SAMPLE_TRAIN_ROWS, seed=SEED + 10, target_col=TARGET_COL)
val_df = cap_split_rows(val_full, SAMPLE_EVAL_ROWS, seed=SEED + 11, target_col=TARGET_COL)
test_df = cap_split_rows(test_full, SAMPLE_EVAL_ROWS, seed=SEED + 12, target_col=TARGET_COL)

split_report = []
for split_name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_report.append({
        'split': split_name,
        'rows': int(len(frame)),
        'fraud_rows': int(frame[TARGET_COL].sum()),
        'fraud_rate': float(frame[TARGET_COL].mean()),
    })

pd.DataFrame(split_report)

## 6) Model helpers + XGBoost baseline

In [ ]:
def evaluate_classifier(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    return {
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'pr_auc': float(average_precision_score(y_true, y_score)),
        'brier': float(brier_score_loss(y_true, y_score)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }


def make_model_xy(frame: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    y = frame[TARGET_COL].to_numpy(dtype=np.int32)
    meta = frame[[c for c in META_COLS if c in frame.columns]].reset_index(drop=True)

    X = frame.drop(columns=[TARGET_COL]).copy()

    # Drop known IDs/high-cardinality fields from modeling.
    for c in DROP_LEAKY_OR_ID:
        if c in X.columns:
            X = X.drop(columns=[c])

    # Keep dates as engineered numeric info only.
    for c in ['incident_date', 'policy_bind_date']:
        if c in X.columns:
            X = X.drop(columns=[c])

    X = X.where(pd.notna(X), np.nan)

    non_numeric_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    for col in non_numeric_cols:
        X[col] = X[col].astype(object)

    return X, y, meta


X_train, y_train, train_meta = make_model_xy(train_df)
X_val, y_val, val_meta = make_model_xy(val_df)
X_test, y_test, test_meta = make_model_xy(test_df)

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

logger.info('Features total={} num={} cat={}', len(X_train.columns), len(num_cols), len(cat_cols))


def train_xgboost_baseline(X: pd.DataFrame, y: np.ndarray) -> Pipeline:
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    scale_pos_weight = max(1.0, n_neg / max(1, n_pos))

    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]), cat_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

    model = XGBClassifier(
        n_estimators=240,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=SEED,
        eval_metric='aucpr',
        tree_method='hist',
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )

    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', model),
    ])
    pipe.fit(X, y)
    return pipe


def get_scores(model: Any, X: pd.DataFrame) -> np.ndarray:
    proba = np.asarray(model.predict_proba(X))
    if proba.ndim != 2 or proba.shape[1] != 2:
        raise ValueError(f'Expected binary predict_proba [N,2], got {proba.shape}')
    return proba[:, 1].astype(float)


t0 = time.time()
xgb_model = train_xgboost_baseline(X_train, y_train)
logger.info('XGBoost trained in {:.1f}s', time.time() - t0)

## 7) TabFM checkpoint management

In [ ]:
TABFM_CHECKPOINT_OVERRIDE = os.getenv('TABFM_CHECKPOINT_PATH', '').strip()
DEFAULT_TABFM_CKPT = MODEL_CACHE_DIR / 'classification' / 'pytorch_model.bin'
TABFM_CKPT_PATH = Path(TABFM_CHECKPOINT_OVERRIDE) if TABFM_CHECKPOINT_OVERRIDE else DEFAULT_TABFM_CKPT


def ensure_tabfm_checkpoint(ckpt_path: Path) -> Path:
    if ckpt_path.exists():
        logger.info('TabFM checkpoint found at {} ({:.1f} MB)', ckpt_path, ckpt_path.stat().st_size / (1024**2))
        return ckpt_path

    logger.warning('TabFM checkpoint missing. Downloading to {} ...', MODEL_CACHE_DIR)
    cmd = [
        shutil.which('tabfm') or 'tabfm',
        'download',
        '--model_path',
        str(MODEL_CACHE_DIR),
    ]
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f'TabFM download failed (exit={result.returncode})\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}'
        )

    if not ckpt_path.exists():
        raise FileNotFoundError(f'Expected checkpoint not found at {ckpt_path} after download')

    logger.info('TabFM checkpoint ready: {} ({:.1f} MB)', ckpt_path, ckpt_path.stat().st_size / (1024**2))
    return ckpt_path


TABFM_CKPT_PATH = ensure_tabfm_checkpoint(TABFM_CKPT_PATH)
TABFM_CKPT_PATH

## 8) Train TabFM (3 variants)

We train:
1. `tabfm_default`
2. `tabfm_ensemble_preset`
3. `tabfm_advanced_custom`

Runtime safety knobs:
- Training context cap: `TABFM_CONTEXT_MAX_ROWS`
- Comparison eval cap: `TABFM_EVAL_MAX_ROWS` (champion still re-scored on full splits)
- `TABFM_FAST_MODE=1` for constrained environments

In [ ]:
def load_tabfm_backbone(device: str) -> Any:
    ckpt_root = TABFM_CKPT_PATH.parent.parent if TABFM_CKPT_PATH.parent.name == 'classification' else TABFM_CKPT_PATH.parent
    return tabfm_v1_0_0.load(
        model_type='classification',
        checkpoint_path=str(ckpt_root),
        device=device,
    )


def pick_tabfm_device(requested: str) -> str:
    if requested.startswith('cuda') and torch.cuda.is_available():
        total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info('Detected GPU memory {:.2f} GiB', total_mem_gb)
        if total_mem_gb < 12:
            logger.warning('GPU memory <12 GiB; using CPU for TabFM stability')
            return 'cpu'
    return requested


def fit_tabfm_variants(X: pd.DataFrame, y: np.ndarray, requested_device: str) -> dict[str, TabFMClassifier]:
    device = pick_tabfm_device(requested_device)

    if len(X) > TABFM_CONTEXT_MAX_ROWS:
        X_fit, _, y_fit, _ = train_test_split(
            X,
            y,
            train_size=TABFM_CONTEXT_MAX_ROWS,
            random_state=SEED,
            stratify=y,
        )
        logger.warning('TabFM context capped: {} -> {} rows', len(X), len(X_fit))
    else:
        X_fit, y_fit = X, y

    batch_size = 1 if device == 'cpu' else 2

    if TABFM_FAST_MODE:
        logger.warning('TABFM_FAST_MODE=1 enabled; reduced estimator complexity active')
        default_estimators = 2
        ensemble_estimators = 2
        advanced_estimators = 2
        advanced_norm_methods = ['none']
        advanced_n_feature_crosses = 0
        advanced_n_svd_features = 0
        advanced_total_svd_pool = 32
    else:
        default_estimators = 4
        ensemble_estimators = 6
        advanced_estimators = 6
        advanced_norm_methods = ['none', 'power', 'quantile_rtdl']
        advanced_n_feature_crosses = 'sqrt'
        advanced_n_svd_features = 'sqrt'
        advanced_total_svd_pool = 128

    models: dict[str, TabFMClassifier] = {}

    m_default = TabFMClassifier(
        model=load_tabfm_backbone(device),
        n_estimators=default_estimators,
        batch_size=batch_size,
        random_state=SEED,
        n_feature_crosses=0,
        n_svd_features=0,
        enable_nnls=False,
        binary_calibration_method=None,
        multiclass_calibration_method=None,
        verbose=False,
    )
    m_default.fit(X_fit, y_fit)
    models['tabfm_default'] = m_default

    m_ensemble = TabFMClassifier.ensemble(
        model=load_tabfm_backbone(device),
        n_estimators=ensemble_estimators,
        batch_size=batch_size,
        random_state=SEED,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        verbose=False,
    )
    m_ensemble.fit(X_fit, y_fit)
    models['tabfm_ensemble_preset'] = m_ensemble

    m_advanced = TabFMClassifier(
        model=load_tabfm_backbone(device),
        n_estimators=advanced_estimators,
        norm_methods=advanced_norm_methods,
        feat_shuffle_method='random',
        class_shift=True,
        permute_categorical=True,
        n_feature_crosses=advanced_n_feature_crosses,
        n_svd_features=advanced_n_svd_features,
        total_svd_pool=advanced_total_svd_pool,
        average_logits=False,
        enable_nnls=False,
        nnls_beta=0.75,
        binary_calibration_method=None,
        random_state=SEED,
        batch_size=batch_size,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        verbose=False,
    )
    m_advanced.fit(X_fit, y_fit)
    models['tabfm_advanced_custom'] = m_advanced

    logger.info('TabFM variants trained on device={}', device)
    return models


t1 = time.time()
tabfm_models = fit_tabfm_variants(X_train, y_train, requested_device=DEVICE)
logger.info('TabFM training done in {:.1f}s', time.time() - t1)

## 9) Evaluate models and choose champion

In [ ]:
def build_eval_slice(X: pd.DataFrame, y: np.ndarray, max_rows: int, seed_offset: int) -> tuple[pd.DataFrame, np.ndarray]:
    if max_rows == 0 or len(X) <= max_rows:
        return X, y

    stratify_target = y if len(np.unique(y)) > 1 else None
    _, X_sampled, _, y_sampled = train_test_split(
        X,
        y,
        test_size=max_rows,
        random_state=SEED + seed_offset,
        stratify=stratify_target,
    )
    return X_sampled, y_sampled


X_val_eval, y_val_eval = build_eval_slice(X_val, y_val, TABFM_EVAL_MAX_ROWS, seed_offset=101)
X_test_eval, y_test_eval = build_eval_slice(X_test, y_test, TABFM_EVAL_MAX_ROWS, seed_offset=202)
logger.info('Evaluation rows used (val/test): {}/{}', len(X_val_eval), len(X_test_eval))

model_registry: dict[str, Any] = {'xgboost_baseline': xgb_model, **tabfm_models}
rows: list[dict[str, Any]] = []
predictions: dict[str, dict[str, np.ndarray]] = {}

for model_name, model in model_registry.items():
    val_scores = get_scores(model, X_val_eval)
    test_scores = get_scores(model, X_test_eval)
    predictions[model_name] = {'val': val_scores, 'test': test_scores}

    rows.append({'model': model_name, 'split': 'val', **evaluate_classifier(y_val_eval, val_scores)})
    rows.append({'model': model_name, 'split': 'test', **evaluate_classifier(y_test_eval, test_scores)})

metrics_df = pd.DataFrame(rows).sort_values(['split', 'pr_auc'], ascending=[True, False]).reset_index(drop=True)
val_rank = metrics_df[metrics_df['split'] == 'val'].sort_values('pr_auc', ascending=False).reset_index(drop=True)
champion_model_name = val_rank.loc[0, 'model']
logger.info('Champion model by validation PR-AUC: {}', champion_model_name)

champion_model = model_registry[champion_model_name]
champion_val_scores = get_scores(champion_model, X_val)
champion_test_scores = get_scores(champion_model, X_test)
logger.info('Champion full scoring rows (val/test): {}/{}', len(champion_val_scores), len(champion_test_scores))

metrics_df

In [ ]:
plot_df = metrics_df[metrics_df['split'] == 'test'].sort_values('pr_auc', ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=plot_df, x='pr_auc', y='model', ax=axes[0], palette='viridis')
axes[0].set_title('Test PR-AUC')
axes[0].set_xlabel('PR-AUC')
axes[0].set_ylabel('')

sns.barplot(data=plot_df, x='roc_auc', y='model', ax=axes[1], palette='mako')
axes[1].set_title('Test ROC-AUC')
axes[1].set_xlabel('ROC-AUC')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
if len(champion_test_scores) != len(y_test):
    raise ValueError(f'Champion score mismatch: len(scores)={len(champion_test_scores)} len(y_test)={len(y_test)}')

prob_true, prob_pred = calibration_curve(y_test, champion_test_scores, n_bins=10, strategy='quantile')

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.plot(prob_pred, prob_true, marker='o', label=champion_model_name)
plt.title('Calibration Curve (Test)')
plt.xlabel('Predicted suspicious-claim probability')
plt.ylabel('Observed suspicious fraction')
plt.legend()
plt.tight_layout()
plt.show()

## 10) Investigation policy optimization

We optimize a review threshold from validation data using expected net savings:

`net = prevented_fraud_payout - review_cost`

- Prevented payout is approximated by reviewed fraudulent claim amount times an assumed capture rate.

In [ ]:
def optimize_review_threshold(
    y_true: np.ndarray,
    y_score: np.ndarray,
    claim_amount: np.ndarray,
    review_cost: float,
    fraud_capture_rate: float,
    grid_size: int = 401,
) -> tuple[pd.DataFrame, dict[str, float]]:
    thresholds = np.linspace(0.0, 1.0, grid_size)
    rows: list[dict[str, float]] = []

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    claim_amount = np.asarray(claim_amount).astype(float)

    for thr in thresholds:
        flagged = (y_score >= thr)
        fraud_flagged = flagged & (y_true == 1)

        prevented_payout = float((claim_amount[fraud_flagged] * fraud_capture_rate).sum())
        review_count = int(flagged.sum())
        total_review_cost = float(review_count * review_cost)
        expected_net_savings = prevented_payout - total_review_cost

        rows.append({
            'threshold': float(thr),
            'review_count': float(review_count),
            'fraud_flagged_count': float(fraud_flagged.sum()),
            'prevented_payout': prevented_payout,
            'review_cost_total': total_review_cost,
            'expected_net_savings': expected_net_savings,
        })

    curve = pd.DataFrame(rows)
    best_idx = int(curve['expected_net_savings'].idxmax())
    return curve, curve.loc[best_idx].to_dict()


BUSINESS_ASSUMPTIONS = {
    'review_cost': 120.0,
    'fraud_capture_rate': 0.65,
}

val_claim_amount = val_meta['total_claim_amount'].to_numpy(dtype=float)
test_claim_amount = test_meta['total_claim_amount'].to_numpy(dtype=float)

threshold_curve_df, best_threshold_row = optimize_review_threshold(
    y_true=y_val,
    y_score=champion_val_scores,
    claim_amount=val_claim_amount,
    **BUSINESS_ASSUMPTIONS,
)
best_threshold = float(best_threshold_row['threshold'])
logger.info('Best review threshold on validation: {:.3f}', best_threshold)

medium_threshold = max(0.0, best_threshold - 0.10)

policy_df = test_meta.copy()
policy_df['is_suspicious'] = y_test
policy_df['suspicion_score'] = champion_test_scores
policy_df['review_action'] = np.select(
    [
        policy_df['suspicion_score'] >= best_threshold,
        policy_df['suspicion_score'] >= medium_threshold,
    ],
    [
        'priority_investigate',
        'secondary_review',
    ],
    default='fast_track_approve',
)

policy_summary = (
    policy_df.groupby('review_action', dropna=False)
    .agg(
        claims=('claim_id', 'count'),
        avg_claim_amount=('total_claim_amount', 'mean'),
        suspicious_rate=('is_suspicious', 'mean'),
    )
    .reset_index()
    .sort_values('claims', ascending=False)
)

policy_summary

In [ ]:
plt.figure(figsize=(8, 4))
sns.lineplot(data=threshold_curve_df, x='threshold', y='expected_net_savings', linewidth=2)
plt.axvline(best_threshold, color='red', linestyle='--', label=f'best={best_threshold:.3f}')
plt.title('Expected Net Savings vs Review Threshold (Validation)')
plt.xlabel('Threshold')
plt.ylabel('Expected Net Savings')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
test_view = test_df[['claim_id', 'incident_severity', 'incident_type', 'auto_year', 'months_as_customer', 'total_claim_amount']].copy()

top_suspicious = (
    policy_df.sort_values('suspicion_score', ascending=False)
    .head(20)
    .merge(test_view, on='claim_id', how='left')
)

top_suspicious

## 11) Persist artifacts

In [ ]:
metrics_path = ARTIFACT_DIR / 'problem4_insurance_model_metrics.csv'
pred_path = ARTIFACT_DIR / 'problem4_insurance_predictions_test.parquet'
policy_path = ARTIFACT_DIR / 'problem4_insurance_review_actions.csv'
policy_summary_path = ARTIFACT_DIR / 'problem4_insurance_policy_summary.csv'
threshold_curve_path = ARTIFACT_DIR / 'problem4_insurance_threshold_curve.csv'
threshold_summary_path = ARTIFACT_DIR / 'problem4_insurance_threshold_summary.csv'
runtime_meta_path = ARTIFACT_DIR / 'problem4_insurance_runtime_meta.json'

prediction_frame = policy_df[[
    'claim_id',
    'is_suspicious',
    'suspicion_score',
    'review_action',
    'total_claim_amount',
]].copy()

metrics_df.to_csv(metrics_path, index=False)
policy_df.to_csv(policy_path, index=False)
policy_summary.to_csv(policy_summary_path, index=False)
threshold_curve_df.to_csv(threshold_curve_path, index=False)

threshold_summary_df = pd.DataFrame([
    {'split': 'validation', **best_threshold_row},
    {
        'split': 'test',
        **optimize_review_threshold(
            y_true=y_test,
            y_score=champion_test_scores,
            claim_amount=test_claim_amount,
            **BUSINESS_ASSUMPTIONS,
        )[1],
    },
])
threshold_summary_df.to_csv(threshold_summary_path, index=False)

prediction_frame_cast = prediction_frame.copy()
prediction_frame_cast['claim_id'] = prediction_frame_cast['claim_id'].astype('int64')
prediction_frame_cast['is_suspicious'] = prediction_frame_cast['is_suspicious'].astype('int32')
prediction_frame_cast['suspicion_score'] = prediction_frame_cast['suspicion_score'].astype('float64')
prediction_frame_cast['review_action'] = prediction_frame_cast['review_action'].astype(str)
prediction_frame_cast['total_claim_amount'] = prediction_frame_cast['total_claim_amount'].astype('float64')

pred_pl = pl.DataFrame({col: prediction_frame_cast[col].tolist() for col in prediction_frame_cast.columns})
pred_pl.write_parquet(pred_path)

runtime_meta = {
    'seed': SEED,
    'data_url': DATA_URL,
    'sample_train_rows': int(SAMPLE_TRAIN_ROWS),
    'sample_eval_rows': int(SAMPLE_EVAL_ROWS),
    'tabfm_context_max_rows': int(TABFM_CONTEXT_MAX_ROWS),
    'tabfm_eval_max_rows': int(TABFM_EVAL_MAX_ROWS),
    'tabfm_fast_mode': bool(TABFM_FAST_MODE),
    'tabfm_device_preference': TABFM_DEVICE_PREF,
    'tabfm_device_effective': DEVICE,
    'tabfm_checkpoint': str(TABFM_CKPT_PATH),
    'champion_model': champion_model_name,
    'best_threshold': best_threshold,
    'business_assumptions': BUSINESS_ASSUMPTIONS,
}
runtime_meta_path.write_text(json.dumps(runtime_meta, indent=2))

for path in [
    metrics_path,
    pred_path,
    policy_path,
    policy_summary_path,
    threshold_curve_path,
    threshold_summary_path,
    runtime_meta_path,
]:
    logger.info('Wrote {}', path)

sorted(p.name for p in ARTIFACT_DIR.glob('problem4_insurance_*'))

## 12) Practical deployment notes

For production rollout:
1. Calibrate thresholds per line-of-business and geography.
2. Add post-decision outcomes (investigator confirmations) for continual retraining.
3. Monitor drift for claim amount distribution, incident mix, and suspicious rate.
4. Use explainability tooling (SHAP + counterfactuals) for investigator trust and compliance.